# Pipeline B - Mid-fusion + painted LiDAR range (ablation of A)

Pipeline A **plus** the LiDAR reprojected into the camera as a sparse range
channel added to the image-space fusion (LiDAR feeds the net twice: painted
in-image + pillarized in BEV). The painted-range map is exactly what
`sample.depth_left` already produces. Toggle it with `USE_PAINTED_RANGE` for a
clean A/B test.

Same structure as A; unbuilt stages left empty.

## 1. Imports

In [1]:
import os
from pathlib import Path
import data  # noqa: F401  (repo module; also used below)

# Point py123d at the repo-local roots: converted logs in data/, raw KITTI-360
# sensor blobs in KITTI-360/. setdefault respects the env vars if you already
# exported them (see README), so this only fills in the gap when running the
# notebook with a bare kernel.
_REPO = Path(data.__file__).resolve().parent
os.environ.setdefault("PY123D_DATA_ROOT", str(_REPO / "data"))
os.environ.setdefault("KITTI360_DATA_ROOT", str(_REPO / "KITTI-360"))

import numpy as np
import torch

# Project modules — the implementation stays inside the .py files (6-file layout:
# data / evaluation / globals / network / train / utils).
import data, utils
import globals as G                       # shared BEV grid + channel contract (single source)
from data import Py123dDataset
from network import PillarConfig, PointPillarsBranch, StereoBEVBranch   # full network

import train        # TargetEncoder / CenterPointLoss / train_model / overfit_one_frame
import evaluation   # CenterPointDecoder + evaluate_model (center-distance AP)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cuda


/home/leonardo/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## 2. Globals

In [2]:
# globals.py is the single source of truth for the shared BEV grid + channel
# contract. Every branch config (PillarConfig, MonoBEVConfig, StereoBEVConfig)
# defaults to it, so the grid can't drift between branches. Indexing: ix*ny+iy.
print("grid_size (nx, ny):", G.GRID_SIZE, "| x_range:", G.X_RANGE, "| y_range:", G.Y_RANGE)
print("channels — camera:", G.CAMERA_BEV_CHANNELS, "| lidar:", G.LIDAR_BEV_CHANNELS,
      "| num_classes:", G.NUM_CLASSES)

# Pipeline B toggle: feed the painted LiDAR-range channel into the image stage.
USE_PAINTED_RANGE = True

grid_size (nx, ny): (200, 160) | x_range: (0.0, 50.0) | y_range: (-20.0, 20.0)
channels — camera: 64 | lidar: 128 | num_classes: 3


## 3. Utils

In [3]:
# Visualisation helpers live in utils.py (LiDAR density BEV + GT boxes, point
# cloud, stereo pair, frustum, voxels, clusters).
import utils
print([f for f in dir(utils) if f.startswith("visualize_")])
# e.g. utils.visualize_bev(sample)  -> ego-frame boxes over a LiDAR BEV

['visualize_bev', 'visualize_clusters', 'visualize_detections', 'visualize_encoded_targets', 'visualize_evaluation', 'visualize_frustum', 'visualize_images', 'visualize_pipeline_debug', 'visualize_pointcloud', 'visualize_stereo_bev_diagnostic', 'visualize_voxels']


## 4. Data

In [4]:
# data.py loads KITTI-360 through py123d and assembles one StereoSample / frame.
dataset = Py123dDataset(split_names=["kitti360_train"], max_num_scenes=1)
frame   = dataset.get_frame(0, dataset.scenes[0].number_of_history_iterations + 13)
sample  = frame.to_stereo_sample()
print(sample)

# GT boxes: global frame (for 2D projection) + ego frame (already converted, for
# BEV target encoding). See data.boxes_global_to_ego / assert_boxes_in_sensor_range.
print("boxes_3d (global):", sample.boxes_3d.shape,
      "| boxes_3d_ego:", sample.boxes_3d_ego.shape)
# utils.visualize_images(sample); utils.visualize_pointcloud(sample)

# Painted-range channel = LiDAR projected into the left image. On KITTI-360 this
# is loaded from preprocessed/depth_maps and is None unless those were generated
# (they are optional). NOTE: the camera branch now uses REAL stereo depth
# (SGBM / IGEV via StereoBEVBranch, image_right is used); painted-range is a
# separate LiDAR-in-image channel, still TODO to wire into the image stage.
print("depth_left (painted range, may be None on KITTI-360):",
      None if sample.depth_left is None else sample.depth_left.shape)

StereoSample(dataset='kitti360', log='2013_05_28_drive_0003_sync', iter=13, boxes_3d=3, boxes_2d=2, lidar_pts=120772, depth=no)
boxes_3d (global): (3, 10) | boxes_3d_ego: (3, 10)
depth_left (painted range, may be None on KITTI-360): None


## 5. Network

In [5]:
# --- Stage A: same two branches as Pipeline A ---------------------------------
lidar_branch  = PointPillarsBranch(PillarConfig()).to(DEVICE).eval()
camera_branch = StereoBEVBranch().to(DEVICE).eval()

# PointPillars wants (N, 4) = [x, y, z, intensity], so append the reflectance.
pts_lidar = np.concatenate(
    [sample.lidar_xyz, sample.lidar_features["intensity"][:, None]], axis=1
).astype(np.float32)
with torch.no_grad():
    bev_lidar  = lidar_branch(pts_lidar, device=DEVICE)         # (C_lidar, nx, ny)
    bev_camera = camera_branch(sample, device=DEVICE)            # (C_cam,   nx, ny)
print("BEV_lidar:", tuple(bev_lidar.shape), "| BEV_camera:", tuple(bev_camera.shape))

# TODO (Pipeline B specific): inject the painted-range channel (sample.depth_left)
#   into the image-space fusion BEFORE the splat, gated by USE_PAINTED_RANGE.
#   Painted range is sparse with few beams -> use sparsity-aware conv. This only
#   changes the camera branch internals; the BEV fusion contract below is identical.

# --- BEV fusion + 2D CenterPoint head (network.py) — same as Pipeline A --------
from network import BEVDetector, describe

detector = BEVDetector.from_bev_maps(bev_camera, bev_lidar, num_classes=3).to(DEVICE).eval()
describe(detector)

with torch.no_grad():
    out = detector(bev_camera, bev_lidar)
print({k: tuple(v.shape) for k, v in out.items()})

# TODO (target encoder): rasterize sample.boxes_3d_ego[:, :2] + class (2D BEV only).

BEV_lidar: (128, 200, 160) | BEV_camera: (64, 200, 160)
BEV fusion contract — Stage A must emit, on the shared grid:
  camera BEV : (B, 64, 200, 160)
  lidar  BEV : (B, 128, 200, 160)
  fused      : (B, 128, 200, 160)
  head out   : heatmap (B, 3, 200, 160) + offset (B, 2, 200, 160)
parameters:
  fusion : 369,152
  head   : 74,181
  total  : 443,333
{'heatmap': (1, 3, 200, 160), 'offset': (1, 2, 200, 160)}


## 6. Train

In [6]:
# train.py is implemented (TargetEncoder / CenterPointLoss / train_model).
# Pipeline B painted-range wiring is still TODO, so PipelineB(use_painted_range=True)
# raises; use training.ipynb with PipelineA as the control arm.

## 7. Test

In [7]:
# evaluation.py is implemented: evaluate_model -> center-distance AP @0.5/1/2/4 m
# per class + print_ap_report. (CDS / per-range bins / Orin latency: TODO.)